In [74]:
import pandas as pd
import matplotlib.pyplot as plt

# 讀取資料
user_data = pd.read_csv('user_data.csv')
dp001_prac = pd.read_csv('dp001_prac.csv')
dp001_exam = pd.read_csv('dp001_exam.csv')
dp001_review = pd.read_csv('dp001_review.csv')
dp001_review_plus = pd.read_csv('dp001_review_plus.csv')

In [69]:
# 選擇需要的欄位
scores = user_data[['user_sn', 'chinese_score', 'math_score', 'english_score']]

# 標準化各科成績
scores[['chinese_score', 'math_score', 'english_score']] = scores[['chinese_score', 'math_score', 'english_score']].apply(lambda x: (x - x.mean()) / x.std())

# 加總各科標準化成績
scores['total_score'] = scores[['chinese_score', 'math_score', 'english_score']].sum(axis=1)

# 排序並取前1/4與後1/4
scores = scores.sort_values(by='total_score', ascending=False)
top_quarter = scores.head(len(scores) // 4).reset_index(drop=True)
bottom_quarter = scores.tail(len(scores) // 4).reset_index(drop=True)

C:\Users\USER\AppData\Local\Temp\ipykernel_27212\118768189.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scores[['chinese_score', 'math_score', 'english_score']] = scores[['chinese_score', 'math_score', 'english_score']].apply(lambda x: (x - x.mean()) / x.std())
C:\Users\USER\AppData\Local\Temp\ipykernel_27212\118768189.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scores['total_score'] = scores[['chinese_score', 'math_score', 'english_score']].sum(axis=1)


將各科的exam_score標準化後，做加總並排序，設分數前1/4為高分組，後1/4為低分組，進行比較

比較兩組之間在影片行為中的不同，以及是否有顯著效果?

In [73]:
# 篩選出指定的 review_sn 列表的資料
filtered_review_data = dp001_review_plus[dp001_review_plus['review_sn']==104741619]

# 計算各 review_sn 中 view_action 的次數與平均秒數
review_action_summary = filtered_review_data.groupby(['review_sn', 'view_action']).agg(
    count=('view_action', 'size'),
    avg_seconds=('timestamp', 'mean')  # 假設 timestamp 欄位表示秒數
).reset_index()
review_action_summary

,review_sn,view_action,count,avg_seconds
0,104741619,browse,1,0.000000
1,104741619,chkptend,6,87.666667
2,104741619,chkptstart,6,87.666667
3,104741619,end,1,141.290000
4,104741619,fuscreenon,1,135.520000
5,104741619,normal,2,65.635000
6,104741619,paused,7,75.475714
7,104741619,play,8,55.353750
8,104741619,review,2,74.500000
9,104741619,slowdown,1,66.960000


整理出影片編號104741619的行為記錄次數及平均秒數

In [30]:
# 定義需要篩選的 review_sn 清單
review_sn_list = [
    104741619, 104744930, 104746418, 104747537, 105374257, 105377782, 105382397, 105384762, 105516833, 106325047, 
    106327527, 106332585, 106336606, 106343203, 107455391, 107458856, 107467689, 108796656, 108799401, 109955587, 
    109956689, 110666166, 110668613, 110668999, 110669164, 110671204, 111572431, 111576846, 111578615, 111578739, 
    111579240, 111586492, 111587562, 111660128, 112590244, 112591507, 112596681, 112607191, 116453273, 116457076, 
    116459744, 116461151, 116462389, 116589503, 116590532, 116591258, 116592370, 117862334, 117867524, 117876026, 
    119389699, 119493271, 119498153, 122084946, 122089076, 122093522, 122365939, 122374350, 122378352, 122380473, 
    122423152, 122431768, 122502278, 122502455, 122502637, 122502704
]

# 過濾出指定的 review_sn 並計算每個 review_sn 的 view_action 次數
review_actions_counts_all = dp001_review_plus[dp001_review_plus['review_sn'].isin(review_sn_list)]
review_action_summary = review_actions_counts_all.groupby(['review_sn', 'view_action']).size().unstack(fill_value=0)
review_action_summary


view_action,browse,chkptend,chkptstart,continue,dragleft,dragright,dragstart,end,fuscreenoff,fuscreenon,normal,paused,play,review,slowdown,speedup
review_sn,,,,,,,,,,,,,,,,
104741619,1,6,6,0,0,0,0,1,0,1,2,7,8,2,1,0
104744930,1,4,4,0,0,0,0,1,0,0,0,5,6,0,0,0
104746418,1,5,5,0,1,0,1,1,0,1,0,6,7,0,0,0
104747537,1,3,3,0,0,0,0,1,0,0,0,3,4,0,0,0
105374257,1,4,4,0,0,0,0,1,0,0,0,4,5,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122431768,1,2,2,0,1,0,1,1,0,0,0,3,4,0,0,0
122502278,1,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0
122502455,1,0,0,0,0,1,3,0,0,0,0,4,5,0,0,1


In [45]:
# Filter out data where review_sn is 104741619 in dp001_review_plus
filtered_review_plus = dp001_review_plus[dp001_review_plus['review_sn'] == 104741619]

# Match corresponding user_sn from dp001_review
merged_data = pd.merge(filtered_review_plus, dp001_review[['user_sn', 'review_sn']], on='review_sn', how='left')

# Select specified columns and drop review_plus_sn
merged_data = merged_data[['user_sn', 'review_sn', 'view_time', 'view_action', 'timestamp', 'turbo']]
merged_data

,user_sn,review_sn,view_time,view_action,timestamp,turbo
0,32737,104741619,2024-02-16 15:33:08,browse,0.00,NaN
1,32737,104741619,2024-02-16 15:34:11,play,0.00,NaN
2,32737,104741619,2024-02-16 15:35:08,paused,56.00,NaN
3,32737,104741619,2024-02-16 15:35:08,chkptstart,56.00,NaN
4,32737,104741619,2024-02-16 15:37:26,chkptend,56.00,NaN
5,32737,104741619,2024-02-16 15:37:29,review,56.00,NaN
6,32737,104741619,2024-02-16 15:37:29,play,0.00,NaN
7,32737,104741619,2024-02-16 15:37:32,paused,2.33,NaN
8,32737,104741619,2024-02-16 15:37:37,play,2.33,NaN
9,32737,104741619,2024-02-16 15:38:30,paused,56.00,NaN


In [46]:
# 轉換 view_time 為 datetime 格式以便計算時間間隔
merged_data['view_time'] = pd.to_datetime(merged_data['view_time'])

# 計算每次動作之間的時間間隔（秒數）
merged_data['time_interval'] = merged_data['view_time'].diff().dt.total_seconds()

merged_data['time_interval'] = merged_data['time_interval'].shift(-1)
merged_data


,user_sn,review_sn,view_time,view_action,timestamp,turbo,time_interval
0,32737,104741619,2024-02-16 15:33:08,browse,0.00,NaN,63.0
1,32737,104741619,2024-02-16 15:34:11,play,0.00,NaN,57.0
2,32737,104741619,2024-02-16 15:35:08,paused,56.00,NaN,0.0
3,32737,104741619,2024-02-16 15:35:08,chkptstart,56.00,NaN,138.0
4,32737,104741619,2024-02-16 15:37:26,chkptend,56.00,NaN,3.0
5,32737,104741619,2024-02-16 15:37:29,review,56.00,NaN,0.0
6,32737,104741619,2024-02-16 15:37:29,play,0.00,NaN,3.0
7,32737,104741619,2024-02-16 15:37:32,paused,2.33,NaN,5.0
8,32737,104741619,2024-02-16 15:37:37,play,2.33,NaN,53.0
9,32737,104741619,2024-02-16 15:38:30,paused,56.00,NaN,0.0


In [52]:
# 針對 user_sn 為 32737 的所有 review_sn 做同樣的處理

# 篩選 user_sn 為 32737 的資料
filtered_review_data = dp001_review_plus[dp001_review_plus['review_sn'].isin(
    dp001_review[dp001_review['user_sn'] == 32737]['review_sn'])]

# 將 view_time 轉換為 datetime 格式
filtered_review_data['view_time'] = pd.to_datetime(filtered_review_data['view_time'])

# 計算每次動作之間的時間間隔
filtered_review_data = filtered_review_data.sort_values(by=['review_sn', 'view_time'])  # 確保排序正確
filtered_review_data['time_interval'] = filtered_review_data.groupby('review_sn')['view_time'].diff().dt.total_seconds()

# 將 time_interval 欄位往前移動一格
filtered_review_data['time_interval'] = filtered_review_data['time_interval'].shift(-1)
filtered_review_data = filtered_review_data.reset_index(drop=True)
filtered_review_data


C:\Users\USER\AppData\Local\Temp\ipykernel_27212\40757817.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_review_data['view_time'] = pd.to_datetime(filtered_review_data['view_time'])


,review_plus_sn,review_sn,view_time,view_action,timestamp,turbo,time_interval
0,1173498289,104741619,2024-02-16 15:33:08,browse,0.00,NaN,63.0
1,1173499634,104741619,2024-02-16 15:34:11,play,0.00,NaN,57.0
2,1173501076,104741619,2024-02-16 15:35:08,paused,56.00,NaN,0.0
3,1173501077,104741619,2024-02-16 15:35:08,chkptstart,56.00,NaN,138.0
4,1173504590,104741619,2024-02-16 15:37:26,chkptend,56.00,NaN,3.0
...,...,...,...,...,...,...,...
1273,1272943644,122502704,2024-05-21 13:00:29,paused,10.37,NaN,0.0
1274,1272943648,122502704,2024-05-21 13:00:29,dragstart,14.75,NaN,44.0
1275,1272945596,122502704,2024-05-21 13:01:13,dragright,329.74,NaN,0.0
1276,1272945603,122502704,2024-05-21 13:01:13,play,329.74,NaN,2.0


In [66]:
# 針對 user_sn 為 32737 的所有 review_sn 做同樣的處理

# 篩選 user_sn 為 32737 的資料
filtered_review_data = dp001_review_plus[dp001_review_plus['review_sn'].isin(
    dp001_review[dp001_review['user_sn'] == 32737]['review_sn'])]

# 將 view_time 轉換為 datetime 格式
filtered_review_data['view_time'] = pd.to_datetime(filtered_review_data['view_time'])

# 計算每次動作之間的時間間隔
filtered_review_data = filtered_review_data.sort_values(by=['review_sn', 'view_time'])  # 確保排序正確
filtered_review_data['time_interval'] = filtered_review_data.groupby('review_sn')['view_time'].diff().dt.total_seconds()

# 將 time_interval 欄位往前移動一格
filtered_review_data['time_interval'] = filtered_review_data['time_interval'].shift(-1)
filtered_review_data = filtered_review_data.reset_index(drop=True)
filtered_review_data

C:\Users\USER\AppData\Local\Temp\ipykernel_27212\1229014296.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_review_data['view_time'] = pd.to_datetime(filtered_review_data['view_time'])


,review_plus_sn,review_sn,view_time,view_action,timestamp,turbo,time_interval
0,1173498289,104741619,2024-02-16 15:33:08,browse,0.00,NaN,63.0
1,1173499634,104741619,2024-02-16 15:34:11,play,0.00,NaN,57.0
2,1173501076,104741619,2024-02-16 15:35:08,paused,56.00,NaN,0.0
3,1173501077,104741619,2024-02-16 15:35:08,chkptstart,56.00,NaN,138.0
4,1173504590,104741619,2024-02-16 15:37:26,chkptend,56.00,NaN,3.0
...,...,...,...,...,...,...,...
1273,1272943644,122502704,2024-05-21 13:00:29,paused,10.37,NaN,0.0
1274,1272943648,122502704,2024-05-21 13:00:29,dragstart,14.75,NaN,44.0
1275,1272945596,122502704,2024-05-21 13:01:13,dragright,329.74,NaN,0.0
1276,1272945603,122502704,2024-05-21 13:01:13,play,329.74,NaN,2.0


In [59]:
# 計算每種行為類型（view_action）的平均持續時間和總次數
action_duration_stats = (
    filtered_review_data
    .groupby('view_action')['time_interval']
    .agg(['mean', 'count'])
    .reset_index()
)
# 調整欄位名稱
action_duration_stats.columns = ['view_action', 'average_time_interval', 'total_count']

# 四捨五入平均持續時間至小數點後兩位
action_duration_stats['average_time_interval'] = action_duration_stats['average_time_interval'].round(2)

# 顯示結果
action_duration_stats

,view_action,average_time_interval,total_count
0,browse,9.06,66
1,chkptend,1.63,166
2,chkptstart,23.30,166
3,continue,17.43,7
4,dragleft,4.67,30
5,dragright,2.78,23
6,dragstart,6.21,63
7,end,NaN,0
8,fuscreenoff,1.50,2
9,fuscreenon,90.50,10


高分組所有人的各行為持續平均時間

In [ ]:
# 計算每種行為類型（view_action）的平均持續時間和總次數
action_duration_stats = (
    filtered_review_data
    .groupby('view_action')['time_interval']
    .agg(['mean', 'count'])
    .reset_index()
)
# 調整欄位名稱
action_duration_stats.columns = ['view_action', 'average_time_interval', 'total_count']

# 四捨五入平均持續時間至小數點後兩位
action_duration_stats['average_time_interval'] = action_duration_stats['average_time_interval'].round(2)

# 顯示結果
action_duration_stats

In [57]:
# 選擇需要的欄位
scores = user_data[['user_sn', 'chinese_score', 'math_score', 'english_score']]

# 標準化各科成績
scores[['chinese_score', 'math_score', 
        'english_score']] = scores[['chinese_score', 'math_score',
         'english_score']].apply(lambda x: (x - x.mean()) / x.std())

# 加總各科標準化成績
scores['total_score'] = scores[['chinese_score',
                                 'math_score', 'english_score']].sum(axis=1)

# 排序並取前1/4
scores = scores.sort_values(by='total_score', ascending=False)
top_quarter = scores.head(len(scores) // 4).reset_index(drop=True)

# 取得高分組的 user_sn 列表
top_quarter_user_sn = top_quarter['user_sn']

# 篩選 dp001_review 和 dp001_review_plus 中符合高分組的資料
filtered_review_data_top = dp001_review_plus[dp001_review_plus['review_sn'].isin(
    dp001_review[dp001_review['user_sn'].isin(top_quarter_user_sn)]['review_sn'])]

# 將 view_time 轉換為 datetime 格式
filtered_review_data_top['view_time'] = pd.to_datetime(filtered_review_data_top['view_time'])

# 計算每次動作之間的時間間隔
filtered_review_data_top = filtered_review_data_top.sort_values(by=['review_sn', 'view_time'])  # 確保排序正確
filtered_review_data_top['time_interval'] = filtered_review_data_top.groupby('review_sn')['view_time'].diff().dt.total_seconds()

# 將 time_interval 欄位往前移動一格
filtered_review_data_top['time_interval'] = filtered_review_data_top['time_interval'].shift(-1)

# 計算每種行為類型的平均持續時間
action_duration_avg_top = filtered_review_data_top.groupby('view_action')['time_interval'].mean().reset_index()
action_duration_avg_top.columns = ['view_action', 'average_time_interval']
action_duration_avg_top

C:\Users\USER\AppData\Local\Temp\ipykernel_27212\1249983172.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scores[['chinese_score', 'math_score', 'english_score']] = scores[['chinese_score', 'math_score', 'english_score']].apply(lambda x: (x - x.mean()) / x.std())
C:\Users\USER\AppData\Local\Temp\ipykernel_27212\1249983172.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scores['total_score'] = scores[['chinese_score', 'math_score', 'english_score']].sum(axis=1)
C:\Users\USER\AppData\Local\Temp\ipyk

,view_action,average_time_interval
0,browse,18.891109
1,chkptend,2.661858
2,chkptstart,19.172087
3,continue,12.574586
4,dragleft,9.240196
5,dragright,5.387692
6,dragstart,4.317801
7,end,0.000000
8,fuscreenoff,18.491071
9,fuscreenon,17.951835


In [58]:
# 確保在篩選時保留 user_sn 資訊
# 重新篩選 dp001_review_plus 並合併 dp001_review 中的 user_sn 資訊

# 重新進行合併篩選過程
filtered_review_data_top = pd.merge(
    dp001_review_plus,
    dp001_review[dp001_review['user_sn'].isin(top_quarter_user_sn)][['user_sn', 'review_sn']],
    on='review_sn',
    how='inner'
)

# 確保 view_time 為 datetime 格式
filtered_review_data_top['view_time'] = pd.to_datetime(filtered_review_data_top['view_time'])

# 計算時間間隔
filtered_review_data_top = filtered_review_data_top.sort_values(by=['user_sn', 'review_sn', 'view_time'])
filtered_review_data_top['time_interval'] = filtered_review_data_top.groupby(['user_sn', 'review_sn'])['view_time'].diff().dt.total_seconds()

# 將 time_interval 欄位往前移動一格
filtered_review_data_top['time_interval'] = filtered_review_data_top['time_interval'].shift(-1)

# 計算每位 user_sn 的每種行為類型平均持續時間
action_duration_avg_each_user = (
    filtered_review_data_top.groupby(['user_sn', 'view_action'])['time_interval']
    .mean()
    .reset_index()
)
action_duration_avg_each_user.columns = ['user_sn', 'view_action', 'average_time_interval']
# 將每位 user_sn 的行為類型及其平均時間間隔進行透視處理，以使每個行為成為一欄
action_duration_avg_pivot = action_duration_avg_each_user.pivot_table(
    index='user_sn',
    columns='view_action',
    values='average_time_interval',
    fill_value=0
)

# 四捨五入至小數點後兩位
action_duration_avg_pivot = action_duration_avg_pivot.round(2)

# 重置欄位名稱層級，便於展示
action_duration_avg_pivot.columns.name = None
action_duration_avg_pivot = action_duration_avg_pivot.reset_index()
action_duration_avg_pivot

,user_sn,browse,chkptend,chkptstart,continue,dragleft,dragright,dragstart,end,fuscreenoff,fuscreenon,normal,note,paused,play,review,slowdown,speedup
0,4561,14.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,23.00,0.00,19.33,53.40,0.00,1.00,0.00
1,11312,10.70,1.88,13.69,7.81,1.00,0.00,14.57,0.0,17.73,11.39,10.63,0.00,3.99,9.62,0.00,0.81,20.17
2,15995,10.05,10.90,25.35,16.12,17.42,34.00,7.02,0.0,13.00,6.89,38.67,0.00,46.60,27.49,2.00,4.00,4.00
3,17422,5.20,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,16.00,0.00,4.00,110.83,0.00,0.00,4.00
4,32737,9.06,1.63,23.30,17.43,4.67,2.78,6.21,0.0,1.50,90.50,10.50,0.00,5.97,46.22,0.00,1.33,20.86
5,50836,5.33,2.27,8.68,7.86,0.90,0.38,5.32,0.0,7.00,11.38,5.06,0.00,6.35,15.60,0.00,1.20,7.33
6,65038,28.01,1.44,14.74,38.76,23.39,14.01,3.60,0.0,46.88,33.07,42.64,0.00,17.84,32.96,0.00,7.43,35.00
7,104242,14.29,2.35,16.67,2.00,1.25,0.00,0.81,0.0,20.00,32.64,23.50,0.00,3.31,17.42,0.14,11.00,0.00
8,106042,10.77,1.97,28.86,7.91,8.93,14.33,5.60,0.0,12.00,27.57,39.44,0.00,28.52,34.29,0.38,3.00,0.00
9,108343,16.15,1.91,10.56,0.00,0.62,0.29,2.59,0.0,47.00,10.83,0.00,0.00,1.76,27.29,0.00,0.00,0.00
